In [2]:
from dotenv import load_dotenv
load_dotenv()  # Load environment variables from .env file
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model_name="gpt-5-mini", temperature=0)
import os

In [3]:
os.path.exists("../2_RAG/chroma_db")  # Check if the Chroma database directory exists

True

## Creating Tools

In [4]:
from langchain.tools import tool

In [5]:
@tool
def duckduckgo_search(query) -> str:
    """
    Use this function to perform a search & to answer questions that require up-to-date information. 
    this is a type of web search tool that can be used to retrieve information from the web.
    """
    from langchain_community.tools import DuckDuckGoSearchRun
    search = DuckDuckGoSearchRun()
    return search.invoke(query)

# duckduckgo_search("What is the capital of France?")


In [6]:
duckduckgo_search.invoke("What is the capital of France?")

C:\Users\jethw\AppData\Local\Temp\ipykernel_12000\1041338007.py:7: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools import DuckDuckGoSearchRun


'Paris is the capital of France and the country\'s largest city, as well as its main center for national government, transport, culture, and international tourism. The city is compact enough to explore by neighborhood, but its major museums and monuments require advance planning. A France eSIM can help with metro directions, timed-entry tickets, and restaurant or museum updates. This overview ... France, a country of northwestern Europe, is historically and culturally among the most important countries in the Western world. It has also played a highly significant role in international affairs for centuries. Its capital is Paris, one of the most important cultural and commercial centers in the world. Paris has been the capital of France since 12th century, Paris has been France\'s political and cultural capital since the medieval period, growing under the Capetian kings from the late 10th century into the national capital by the late 12th century. Paris is France\'s largest city, and wi

In [7]:
@tool
def wikipedia_search(query) -> str:
    """
    Use this function to perform a search & to answer questions related to person or a place.
    """
    from langchain_community.tools import WikipediaQueryRun
    from langchain_community.utilities import WikipediaAPIWrapper

    wiki = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper())
    return wiki.invoke(query)

In [8]:
import sys
print(sys.executable)

w:\AI Training\AI Projects\AI_Fundamentals\.venv\Scripts\python.exe


In [9]:
# wikipedia_search.invoke("Sam Altman")

In [10]:
@tool
def arxiv_search(query: str) -> str:
    """Search arXiv for recent research papers matching the query."""
    import arxiv

    search = arxiv.Search(
        query=query,
        max_results=5,
        sort_by=arxiv.SortCriterion.SubmittedDate,
        sort_order=arxiv.SortOrder.Descending,
    )
    papers = list(arxiv.Client().results(search))
    if not papers:
        return f"No arXiv papers found for: {query}"

    return "\n\n".join(
        f"Title: {paper.title}\n"
        f"Authors: {', '.join(author.name for author in paper.authors)}\n"
        f"Published: {paper.published:%Y-%m-%d}\n"
        f"Summary: {paper.summary}\n"
        f"URL: {paper.entry_id}"
        for paper in papers
    )

In [11]:
arxiv_search.invoke("What is the latest research on AI?")

"Title: Towards Physically Grounded Articulated 3D Assets: A Multi-Agent Pipeline and Benchmark\nAuthors: Xudong Cai, Ruijie Zhu, Haoliang Shang, Ruining Li, Marc Pollefeys, Shangzhe Wu, Andrea Vedaldi, Chuanxia Zheng\nPublished: 2026-10-09\nSummary: In this paper, we study physically grounded articulated 3D asset generation---moving the goal from reproducing how objects look to capturing how they move and function. Inspired by the literature on procedural and agentic generation, we present Artisan, a system that extends the latest method Articraft by introducing a multi-agent pipeline with physical simulation feedback, enabling comprehensive attributes beyond geometry, appearance, and physically plausible articulation, such as collision-free motion, stability under gravity, and accurate mass and material properties. Furthermore, we construct a unified benchmark spanning four key aspects: spatial fidelity, physical plausibility, downstream task performance, and overall quality. The ben

In [12]:
@tool
def tool_rag(query: str) -> str:
    """
    Use this tool to answer questions based on NovaS.pdf, which contains information about NovaSphere Technology. 
    This tool retrieves relevant information from the PDF to provide accurate answers to user queries.
    """
    #1. build connection to the Chroma database
    from langchain_community.vectorstores import Chroma
    from langchain_openai import OpenAIEmbeddings
    embed_model = OpenAIEmbeddings(model="text-embedding-3-small")
    chroma_db_conn = Chroma(persist_directory="./chroma_db_semantic", embedding_function=embed_model)

    #2. retrieve relevant documents from the Chroma database based on the query
    relevant_docs = chroma_db_conn.similarity_search(query, k=2)  # Retrieve top 2 relevant documents
    resultant_content = "\n".join([doc.page_content for doc in relevant_docs])  # Combine the content of the retrieved documents
    return resultant_content  # Return the combined content as the answer to the query

In [13]:
tool_rag.invoke("What is NovaSphere Technology?")

C:\Users\jethw\AppData\Local\Temp\ipykernel_12000\3148993785.py:11: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  chroma_db_conn = Chroma(persist_directory="./chroma_db_semantic", embedding_function=embed_model)


'NovaSphere Technologies is a fictional organization created to represent a modern data \nand technology company that has grown gradually over the years. The organization was \nfounded in 2016 by a small group of software engineers who strongly believed that data \nwould become one of the most valuable assets for every business in the future.\nNovaSphere Technologies is a fictional organization created to represent a modern data \nand technology company that has grown gradually over the years. The organization was \nfounded in 2016 by a small group of software engineers who strongly believed that data \nwould become one of the most valuable assets for every business in the future.'

## Tool Binding

In [14]:
## Binding the tools to the LLM

#create a list of tools to bind to the LLM
tool_kit = [duckduckgo_search, wikipedia_search, arxiv_search, tool_rag]
#bind the tools to the LLM
llm_with_tools = llm.bind_tools(tool_kit)

In [15]:
llm_with_tools.invoke("Sam Altman is the CEO of which company?, make use of the tools")

AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 89, 'prompt_tokens': 282, 'total_tokens': 371, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 64, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EY88u6JunerGwa9jjDhGCDJQC3pzg', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a12fca-70ce-7432-8b56-4e031f0b50b8-0', tool_calls=[{'name': 'wikipedia_search', 'args': {'query': 'Sam Altman'}, 'id': 'call_CqbYgXosp4iKhXr7dVDDDarL', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 282, 'output_tokens': 89, 'total_tokens': 371, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, '

In [16]:
## Creating a React Agent with the LLM and the tools
from langchain.agents import create_agent
agent = create_agent(llm_with_tools, tool_kit)

# Call the agent with a query
agent.invoke({"messages": [{"role": "user", "content": "Sam Altman is the CEO of which company?, make use of the tools"}]})

{'messages': [HumanMessage(content='Sam Altman is the CEO of which company?, make use of the tools', additional_kwargs={}, response_metadata={}, id='f3320229-1b71-4278-8ce5-f1de94006a77'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 221, 'prompt_tokens': 282, 'total_tokens': 503, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 192, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EY88wVvtP7MGKUBn4q4U0aC1PHALd', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a12fca-7a92-79b1-9e32-c9e57de2e433-0', tool_calls=[{'name': 'wikipedia_search', 'args': {'query': 'Sam Altman CEO of which company'}, 'id': 'call_8ikRWA1Ze2mvIZp3VDxj0sbn', 'ty

In [17]:
agent.invoke({"messages": [{"role": "user", "content": "When was the NovaSphere Technology founded?"}]})

{'messages': [HumanMessage(content='When was the NovaSphere Technology founded?', additional_kwargs={}, response_metadata={}, id='be65682b-2776-414e-8487-dbf629e84270'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 30, 'prompt_tokens': 275, 'total_tokens': 305, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EY896vRNWEZWDFXE7oymXJk6IoVtb', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a12fca-a185-7f43-96de-99ebbfbbf156-0', tool_calls=[{'name': 'tool_rag', 'args': {'query': 'When was NovaSphere Technology founded?'}, 'id': 'call_zo03zrObkF91HZtBcmIAejkA', 'type': 'tool_call'}], in

## Memory

In [37]:
# agent.invoke({"messages": [{"role": "user", "content": "Could you please summarize this?"}]})
agent_memory = []

def agent_with_memory(query: str):

    #basic step to invoke the agent with the query
    response = agent.invoke({"messages": [{"role": "user", "content": f"Answer this query: {query}. Refer the previous responses for context: \
                                           {str(agent_memory)}"}]})\

    #append this response to the agent's memory
    agent_memory.append(response['messages'])

    #return the agent's response
    return response
    
    # # Append the user query to the agent's memory
    # agent_memory.append({"role": "user", "content": query})
    
    # # Invoke the agent with the current memory
    # response = agent.invoke({"messages": agent_memory})
    
    # # Append the agent's response to the memory
    # agent_memory.append({"role": "assistant", "content": response["output"]})
    
    # return response["output"]

In [38]:
agent_with_memory("When was the NovaSphere Technology founded?")

{'messages': [HumanMessage(content='Answer this query: When was the NovaSphere Technology founded?. Refer the previous responses for context:                                            []', additional_kwargs={}, response_metadata={}, id='b577a3ce-0a3b-4f47-adb5-47232a22acc1'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 93, 'prompt_tokens': 288, 'total_tokens': 381, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 64, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EY95IQKGGV2jqXduASO2Mbpk9wMlI', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a13001-b186-7362-bfe8-2350ccaf5aab-0', tool_calls=[{'name': 'tool_rag', 'args': {'query':

In [39]:
agent_memory

[[HumanMessage(content='Answer this query: When was the NovaSphere Technology founded?. Refer the previous responses for context:                                            []', additional_kwargs={}, response_metadata={}, id='b577a3ce-0a3b-4f47-adb5-47232a22acc1'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 93, 'prompt_tokens': 288, 'total_tokens': 381, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 64, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EY95IQKGGV2jqXduASO2Mbpk9wMlI', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a13001-b186-7362-bfe8-2350ccaf5aab-0', tool_calls=[{'name': 'tool_rag', 'args': {'query': 'When was N

In [40]:
agent_with_memory("Could you please summarize this?")

{'messages': [HumanMessage(content="Answer this query: Could you please summarize this?. Refer the previous responses for context:                                            [[HumanMessage(content='Answer this query: When was the NovaSphere Technology founded?. Refer the previous responses for context:                                            []', additional_kwargs={}, response_metadata={}, id='b577a3ce-0a3b-4f47-adb5-47232a22acc1'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 93, 'prompt_tokens': 288, 'total_tokens': 381, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 64, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EY95IQKGGV2jqXduASO2Mbpk9wMlI', 'service_tier': 'def

In [41]:
agent_memory

[[HumanMessage(content='Answer this query: When was the NovaSphere Technology founded?. Refer the previous responses for context:                                            []', additional_kwargs={}, response_metadata={}, id='b577a3ce-0a3b-4f47-adb5-47232a22acc1'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 93, 'prompt_tokens': 288, 'total_tokens': 381, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 64, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EY95IQKGGV2jqXduASO2Mbpk9wMlI', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a13001-b186-7362-bfe8-2350ccaf5aab-0', tool_calls=[{'name': 'tool_rag', 'args': {'query': 'When was N

In [ ]:
#it will act as a bot where we can ask questions and the agent will respond as long as the user does not type 'exit' to quit the loop
while True:
    user_input = input("Enter your query (or type 'exit' to quit): ")
    if user_input.lower() == 'exit':
        break
    response = agent_with_memory(user_input)
    print("Agent Response:", response['messages'])

Agent Response: [HumanMessage(content='Answer this query: what is NovaSphere Technology. Refer the previous responses for context:                                            [[HumanMessage(content=\'Answer this query: When was the NovaSphere Technology founded?. Refer the previous responses for context:                                            []\', additional_kwargs={}, response_metadata={}, id=\'b577a3ce-0a3b-4f47-adb5-47232a22acc1\'), AIMessage(content=\'\', additional_kwargs={\'refusal\': None}, response_metadata={\'token_usage\': {\'completion_tokens\': 93, \'prompt_tokens\': 288, \'total_tokens\': 381, \'completion_tokens_details\': {\'accepted_prediction_tokens\': 0, \'audio_tokens\': 0, \'reasoning_tokens\': 64, \'rejected_prediction_tokens\': 0}, \'prompt_tokens_details\': {\'audio_tokens\': 0, \'cache_write_tokens\': None, \'cached_tokens\': 0}}, \'model_provider\': \'openai\', \'model_name\': \'gpt-5-mini-2025-08-07\', \'system_fingerprint\': None, \'id\': \'chatcmpl-EY95I